# Music Annotation Project
## Notebook 74: Mask Authority, Annotation Distance and Print-Loss Audit

**Runtime: CPU. Read-only diagnostic.**

This notebook tests whether correction damage is explained by:

1. explicit radius-2 dilation beyond the reviewed seed mask;
2. excess width already present inside the reviewed seed mask;
3. distance from observable annotation evidence;
4. route-specific treatment inside the prepared mask.

It does not alter masks, thresholds, V7.8/V8.8 routes, model probabilities or corrected pages.

### Authoritative HITL lineage

The undilated seed is resolved from the matching HITL review audit `corrected_mask_output` using the original source MD5. The confidence map is independently resolved from the matching draft audit `confidence_map`, reproducing the established Notebook 09 lineage. No erosion or filename-based mask guessing is used.

### Geometry definitions

- **Seed mask**: undilated reviewed/corrected annotation mask.
- **Prepared mask**: frozen dilation-2 intervention mask.
- **Seed** / **Ring 1** / **Ring 2**: exact morphological authority bands.
- **Prepared-mask depth**: Euclidean distance from a prepared-mask pixel to its boundary.
- **Annotation confidence cores**: frozen confidence-map pixels at configurable 0.50, 0.80 and 0.95 levels, restricted to the prepared mask.
- **Dark-seed proxy**: observed dark material inside the undilated seed mask. This is a fallback annotation proxy, not a truth label.
- **Truth annotation core, optional**: on paired development pages only, observed dark material that is not supported by registered clean printed ink. This is post-hoc diagnostic evidence and never an inference input.

### Main visual outputs

- whole-page overlays showing seed, dilation rings, changed pixels and annotation evidence;
- route-by-authority overlays;
- distance-to-annotation heatmaps;
- two-dimensional heatmaps crossing authority band with annotation distance;
- worst-page contact sheets with observed, corrected, geometry and optional truth diagnostics;
- focused crops around changed printed-looking structures.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib seaborn pillow

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, sys, platform, zipfile, gc
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NOTEBOOK72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
SOURCE_RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'notebook74_mask_annotation_distance_audit'

# Authoritative HITL lineage. Notebook 09 resolves reviewed masks and confidence maps
# by the original source MD5 recorded in these audit directories.
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
CLEAN_DIRS=[ROOT/'source'/'clean']

USE_OPTIONAL_CLEAN_TRUTH=True
CONFIDENCE_LEVELS=[0.50,0.80,0.95]
DARK_SEED_THRESHOLDS=[160,190,220]
PRIMARY_DARK_THRESHOLD=190
ANNOTATION_DISTANCE_BINS=[-0.001,1,2,4,8,16,32,np.inf]
DEPTH_BINS=[-0.001,1,2,4,8,16,32,np.inf]
CHANGE_THRESHOLD=8
PRINT_LIKE_VALUE_THRESHOLD=200
MAX_CONTACT_PAGES=16
MAX_FOCUS_CROPS=36
CROP_MARGIN=80

REQUIRED=['prediction_lock.json','notebook72_verdict.json','inputs/runtime_roster.csv','tables/page_output_hashes.csv','pages']
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
SOURCE=Path(SOURCE_RUN_OVERRIDE) if SOURCE_RUN_OVERRIDE else latest_complete(NOTEBOOK72_ROOT,REQUIRED)
if SOURCE is None:raise RuntimeError('No complete Notebook 72 run found.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook74_mask_annotation_distance_audit'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; PAGES=OUT/'pages'; VISUAL=OUT/'visuals'; CROPS=OUT/'focus_crops'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,PAGES,VISUAL,CROPS,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=False)
print('Source:',SOURCE);print('Output:',OUT)

## 2. Integrity helpers and source verification

In [ ]:
def require(x,msg):
    if not bool(x):raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def atomic_json(path,x):
    t=Path(str(path)+'.tmp');t.write_text(json.dumps(x,indent=2,sort_keys=True,default=str));t.replace(path)
def read_gray(path):
    a=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE);require(a is not None,f'Unreadable image: {path}');return a
def binary(a):return (a>0)
def image_index(dirs):
    out={}
    for folder in dirs:
        if not Path(folder).exists():continue
        for p in Path(folder).rglob('*'):
            if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp','.npy'}:
                out.setdefault(p.stem,p)
    return out

def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def read_audit_records(folder,label):
    records=[];failures=[]
    for path in sorted(Path(folder).glob('*.json')):
        try:
            value=json.loads(path.read_text())
            if not isinstance(value,dict):raise ValueError('audit root must be an object')
            value['_audit_path']=str(path);value['_audit_type']=label;records.append(value)
        except Exception as e:failures.append({'audit_type':label,'audit_path':str(path),'error':repr(e)})
    return records,failures

def matches_by_hash(records,source_md5):
    return [r for r in records if str(r.get('source_md5','')).lower()==source_md5.lower()]

def required_audit_path(record,key,audit_type):
    raw=record.get(key)
    require(bool(raw),f'{audit_type} audit {record.get("_audit_path")} missing {key}.')
    p=Path(raw);require(p.exists(),f'{audit_type} audit references missing {key}: {p}')
    return p

def distance_to_core(core):
    # Distance from every pixel to the nearest core pixel. Exact core pixels have distance 0.
    if not np.any(core):return np.full(core.shape,np.nan,np.float32)
    return cv2.distanceTransform((~core).astype(np.uint8),cv2.DIST_L2,5).astype(np.float32)
def depth_inside(mask):return cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5).astype(np.float32)
def printed_ink(clean):return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def band_label(values,bins,prefix=''):
    labels=[]
    for lo,hi in zip(bins[:-1],bins[1:]):labels.append(f'{prefix}{int(max(lo,0))}-{"inf" if np.isinf(hi) else int(hi)}')
    return pd.cut(values,bins=bins,labels=labels,include_lowest=True,right=False)

lock=json.loads((SOURCE/'prediction_lock.json').read_text()); verdict=json.loads((SOURCE/'notebook72_verdict.json').read_text())
require(lock.get('state')=='CLOSED','Notebook 72 prediction lock is not closed.')
require(verdict.get('first_failed_gate') is None,'Notebook 72 has a failed gate.')
roster=pd.read_csv(SOURCE/'inputs/runtime_roster.csv',dtype=str).fillna('')
hashes=pd.read_csv(SOURCE/'tables/page_output_hashes.csv',dtype=str).fillna('').set_index('page_key')
require(lock['payload']['runtime_roster_sha256']==sha256_file(SOURCE/'inputs/runtime_roster.csv'),'Roster changed after lock.')
require(lock['payload']['page_output_hashes_sha256']==sha256_file(SOURCE/'tables/page_output_hashes.csv'),'Hash table changed after lock.')
clean_index=image_index(CLEAN_DIRS)
REVIEW_AUDITS,review_parse_failures=read_audit_records(REVIEW_AUDIT_DIR,'review')
DRAFT_AUDITS,draft_parse_failures=read_audit_records(DRAFT_AUDIT_DIR,'draft')
parse_failures=review_parse_failures+draft_parse_failures
if parse_failures:
    pd.DataFrame(parse_failures).to_csv(AUDIT/'hitl_audit_parse_failures.csv',index=False)
    raise RuntimeError(f'HITL audit parsing failed for {len(parse_failures)} record(s).')
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'source_run':str(SOURCE),'prediction_lock_id':lock['lock_id'],'pages':len(roster),'confidence_levels':CONFIDENCE_LEVELS,'dark_seed_thresholds':DARK_SEED_THRESHOLDS,'change_threshold':CHANGE_THRESHOLD,'clean_truth_optional':USE_OPTIONAL_CLEAN_TRUTH}
atomic_json(INPUTS/'source_identity.json',identity);print(json.dumps(identity,indent=2))

## 3. Input resolution and page-local geometry

The notebook processes one page at a time. It never retains the complete set of image arrays in RAM.

Seed/confidence resolution is explicit and fail-closed where required:

- seed mask is required;
- confidence map is optional and reported as unavailable rather than silently synthesised;
- clean truth is optional and used only for post-hoc diagnostics.

In [ ]:
def resolve_hitl_lineage(roster_row,key,page,prepared):
    source=Path(str(roster_row.source_path))
    require(source.exists(),f'{key}: Notebook 72 source_path missing: {source}')
    source_md5=md5_file(source)
    reviews=matches_by_hash(REVIEW_AUDITS,source_md5)
    drafts=matches_by_hash(DRAFT_AUDITS,source_md5)
    require(len(reviews)<=1,f'{key}: duplicate review audits for source_md5 {source_md5}')
    require(len(drafts)<=1,f'{key}: duplicate draft audits for source_md5 {source_md5}')
    require(len(reviews)==1,f'{key}: no reviewed HITL audit for source_md5 {source_md5}')
    review=reviews[0]
    draft=drafts[0] if len(drafts)==1 else None

    seed_path=required_audit_path(review,'corrected_mask_output','review')
    seed=read_gray(seed_path)>0
    require(seed.shape==prepared.shape,f'{key}: reviewed undilated seed shape {seed.shape} != prepared shape {prepared.shape}')

    # The review audit is authoritative for the undilated reviewed mask. Notebook 09
    # then applies prepare_mask(seed, 2), so this comparison directly audits dilation lineage.
    confidence=None;confidence_path=None;draft_audit_path=None
    if draft is not None:
        draft_audit_path=draft['_audit_path']
        confidence_path=required_audit_path(draft,'confidence_map','draft')
        confidence=np.load(confidence_path).astype(np.float32)
        require(confidence.shape==prepared.shape,f'{key}: confidence shape mismatch')
        if np.nanmax(confidence)>1.00001:confidence=confidence/255.0
        confidence=np.clip(confidence,0,1)

    return {
        'source_path':source,'source_md5':source_md5,
        'seed':seed,'seed_path':seed_path,
        'review_audit_path':Path(review['_audit_path']),
        'confidence':confidence,'confidence_path':confidence_path,
        'draft_audit_path':Path(draft_audit_path) if draft_audit_path else None,
    }

def resolve_clean(key,page,source_dir,shape):
    if not USE_OPTIONAL_CLEAN_TRUTH:return None,None
    candidates=[source_dir/'clean.png']+[p for stem,p in clean_index.items() if stem in {page,key} or stem.startswith(page+'_')]
    for p in candidates:
        if p.exists():
            a=read_gray(p);require(a.shape==shape,f'{key}: clean shape mismatch at {p}');return a,p
    return None,None

def page_inputs(key,page,roster_row=None):
    if roster_row is None:
        matches=roster[(roster.population.astype(str)+'__'+roster.page.astype(str))==key]
        require(len(matches)==1,f'{key}: cannot uniquely recover Notebook 72 roster row')
        roster_row=matches.iloc[0]
    d=SOURCE/'pages'/key
    observed=read_gray(d/'observed.png');corrected=read_gray(d/'corrected.png');baseline=read_gray(d/'baseline_v78.png');prepared=read_gray(d/'prepared_mask.png')>0
    reason=read_gray(d/'reason_map.png');safe=read_gray(d/'safe_remove.png')>0;m3=read_gray(d/'m3.png')>0;m5=read_gray(d/'m5_remove.png')>0
    hitl=resolve_hitl_lineage(roster_row,key,page,prepared)
    seed=hitl['seed'];seed_path=hitl['seed_path'];confidence=hitl['confidence'];confidence_path=hitl['confidence_path']
    clean,clean_path=resolve_clean(key,page,d,observed.shape)
    require(np.all(seed<=prepared),f'{key}: undilated reviewed seed is not a subset of Notebook 72 prepared mask.')

    k1=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
    k2=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
    expected1=cv2.dilate(seed.astype(np.uint8),k1)>0
    expected2=cv2.dilate(seed.astype(np.uint8),k2)>0
    dilation_xor=int(np.logical_xor(prepared,expected2).sum())
    ring1=expected1&~seed
    ring2=expected2&~expected1
    route={'preserve':reason==1,'strict_print':reason==2,'m5':reason==3,'safe_remove':reason==4}
    return locals()

resolution=[]
for _,r in roster.iterrows():
    key=f'{r.population}__{r.page}'
    try:
        z=page_inputs(key,str(r.page),r)
        resolution.append({
            'page_key':key,'page':str(r.page),'population':str(r.population),
            'source_path':str(z['hitl']['source_path']),'source_md5':z['hitl']['source_md5'],
            'seed_path':str(z['seed_path']),'review_audit_path':str(z['hitl']['review_audit_path']),
            'confidence_path':str(z['confidence_path']) if z['confidence_path'] else None,
            'draft_audit_path':str(z['hitl']['draft_audit_path']) if z['hitl']['draft_audit_path'] else None,
            'clean_path':str(z['clean_path']) if z['clean_path'] else None,
            'seed_pixels':int(z['seed'].sum()),'prepared_pixels':int(z['prepared'].sum()),
            'ring1_pixels':int(z['ring1'].sum()),'ring2_pixels':int(z['ring2'].sum()),
            'prepared_vs_exact_dilation2_xor':z['dilation_xor'],
            'confidence_available':z['confidence'] is not None,'clean_available':z['clean'] is not None,
            'seed_sha256':sha256_file(z['seed_path']),
            'review_audit_sha256':sha256_file(z['hitl']['review_audit_path']),
            'confidence_sha256':sha256_file(z['confidence_path']) if z['confidence_path'] else None,
            'draft_audit_sha256':sha256_file(z['hitl']['draft_audit_path']) if z['hitl']['draft_audit_path'] else None,
        })
        del z;gc.collect()
    except Exception as e:
        resolution.append({'page_key':key,'page':str(r.page),'population':str(r.population),'error':repr(e)})
resolution=pd.DataFrame(resolution);resolution.to_csv(AUDIT/'input_resolution.csv',index=False)
if 'error' in resolution.columns and resolution.error.notna().any():
    display(resolution[resolution.error.notna()]);raise RuntimeError('HITL lineage resolution failed. See audit/input_resolution.csv')
print('Reviewed seeds resolved:',resolution.seed_path.notna().sum(),'/',len(resolution))
print('Confidence maps resolved:',int(resolution.confidence_available.sum()),'/',len(resolution))
print('Clean references resolved:',int(resolution.clean_available.sum()),'/',len(resolution))
print('Pages where Notebook 72 prepared mask != exact dilation-2 of HITL reviewed seed:',int((resolution.prepared_vs_exact_dilation2_xor>0).sum()))

## 4. Build partitioned pixel audit and page summaries

In [ ]:
PARTS=TABLES/'pixel_parts';PARTS.mkdir(exist_ok=False)
page_rows=[];cell_rows=[];overall_rows=[]
for _,r in roster.iterrows():
    key=f'{r.population}__{r.page}';page=str(r.page);pop=str(r.population);z=page_inputs(key,page)
    observed=z['observed'];corrected=z['corrected'];baseline=z['baseline'];prepared=z['prepared'];seed=z['seed'];ring1=z['ring1'];ring2=z['ring2'];confidence=z['confidence'];clean=z['clean'];route=z['route'];safe=z['safe'];m5=z['m5']
    authority=np.full(observed.shape,'OUTSIDE',object);authority[seed]='SEED';authority[ring1]='RING1';authority[ring2]='RING2';authority[prepared&~(seed|ring1|ring2)]='OTHER_PREPARED'
    prepared_depth=depth_inside(prepared);seed_depth=depth_inside(seed)
    delta=corrected.astype(np.int16)-observed.astype(np.int16);abs_change=np.abs(delta);changed=prepared&(abs_change>=CHANGE_THRESHOLD);whitened=prepared&(delta>=CHANGE_THRESHOLD)
    # Observable annotation proxies.
    dark_cores={t:(seed&(observed<t)) for t in DARK_SEED_THRESHOLDS}
    dark_dist={t:distance_to_core(core) for t,core in dark_cores.items()}
    conf_cores={};conf_dist={}
    for t in CONFIDENCE_LEVELS:
        core=(prepared&(confidence>=t)) if confidence is not None else np.zeros(prepared.shape,bool)
        conf_cores[t]=core;conf_dist[t]=distance_to_core(core)
    # Optional truth-side annotation core for development diagnostics only.
    truth_annotation_core=None;truth_annotation_distance=np.full(prepared.shape,np.nan,np.float32);truth_print=np.zeros(prepared.shape,bool)
    if clean is not None:
        truth_print=cv2.dilate(printed_ink(clean).astype(np.uint8),np.ones((3,3),np.uint8))>0
        truth_annotation_core=seed&(observed<PRIMARY_DARK_THRESHOLD)&~truth_print
        truth_annotation_distance=distance_to_core(truth_annotation_core)
    # Printed-looking observable evidence. This is descriptive, not truth.
    edges=cv2.Canny(observed,70,170)>0
    print_like=(observed<PRINT_LIKE_VALUE_THRESHOLD)&(edges|route['preserve']|route['strict_print'])
    ys,xs=np.where(prepared)
    frame=pd.DataFrame({'page_key':key,'page':page,'population':pop,'y':ys.astype(np.int32),'x':xs.astype(np.int32),'authority_band':authority[ys,xs].astype(str),'prepared_depth':prepared_depth[ys,xs].astype(np.float32),'seed_depth':seed_depth[ys,xs].astype(np.float32),'observed':observed[ys,xs].astype(np.uint8),'corrected':corrected[ys,xs].astype(np.uint8),'baseline_v78':baseline[ys,xs].astype(np.uint8),'signed_change':delta[ys,xs].astype(np.int16),'absolute_change':abs_change[ys,xs].astype(np.uint16),'changed_8plus':changed[ys,xs],'whitened_8plus':whitened[ys,xs],'print_like_observable':print_like[ys,xs],'route':np.select([route['preserve'][ys,xs],route['strict_print'][ys,xs],safe[ys,xs],m5[ys,xs]],['V78_PRESERVE','STRICT_PRINT','SAFE_REMOVE','M5_REMOVE'],default='OTHER'),'distance_to_dark_seed_190':dark_dist[PRIMARY_DARK_THRESHOLD][ys,xs].astype(np.float32),'truth_print_posthoc':truth_print[ys,xs] if clean is not None else False,'distance_to_truth_annotation_posthoc':truth_annotation_distance[ys,xs].astype(np.float32)})
    for t in DARK_SEED_THRESHOLDS:frame[f'distance_to_dark_seed_{t}']=dark_dist[t][ys,xs].astype(np.float32)
    for t in CONFIDENCE_LEVELS:frame[f'distance_to_conf_{int(t*100):02d}']=conf_dist[t][ys,xs].astype(np.float32)
    frame['annotation_distance_band']=band_label(frame.distance_to_dark_seed_190,ANNOTATION_DISTANCE_BINS)
    frame['prepared_depth_band']=band_label(frame.prepared_depth,DEPTH_BINS,'D')
    frame.to_parquet(PARTS/f'{key}.parquet',index=False)
    # Two-dimensional summaries.
    for (ab,db),q in frame.groupby(['authority_band','annotation_distance_band'],observed=True):
        cell_rows.append({'page_key':key,'population':pop,'authority_band':ab,'annotation_distance_band':str(db),'pixels':len(q),'changed':int(q.changed_8plus.sum()),'whitened':int(q.whitened_8plus.sum()),'print_like_pixels':int(q.print_like_observable.sum()),'changed_print_like':int((q.changed_8plus&q.print_like_observable).sum()),'whitened_print_like':int((q.whitened_8plus&q.print_like_observable).sum()),'mean_whitening':float(q.signed_change.clip(lower=0).mean()),'mean_abs_change':float(q.absolute_change.mean())})
    page_rows.append({'page_key':key,'page':page,'population':pop,'prepared_pixels':int(prepared.sum()),'seed_pixels':int(seed.sum()),'ring1_pixels':int(ring1.sum()),'ring2_pixels':int(ring2.sum()),'changed_pixels':int(changed.sum()),'whitened_pixels':int(whitened.sum()),'changed_print_like':int((changed&print_like).sum()),'whitened_print_like':int((whitened&print_like).sum()),'seed_whitened_print_like':int((whitened&print_like&seed).sum()),'ring1_whitened_print_like':int((whitened&print_like&ring1).sum()),'ring2_whitened_print_like':int((whitened&print_like&ring2).sum()),'mean_distance_changed_print_like_to_dark_seed':float(dark_dist[PRIMARY_DARK_THRESHOLD][changed&print_like].mean()) if np.any(changed&print_like) else np.nan,'confidence_available':confidence is not None,'clean_available':clean is not None})
    del z,frame,observed,corrected,baseline,prepared,seed,ring1,ring2,confidence,clean,authority,prepared_depth,seed_depth,delta,abs_change,changed,whitened,dark_cores,dark_dist,conf_cores,conf_dist,truth_annotation_core,truth_annotation_distance,truth_print,edges,print_like,ys,xs
    gc.collect()
page_df=pd.DataFrame(page_rows);cells_df=pd.DataFrame(cell_rows)
page_df.to_csv(TABLES/'page_summary.csv',index=False);cells_df.to_csv(TABLES/'authority_by_annotation_distance_by_page.csv',index=False)
pooled=cells_df.groupby(['authority_band','annotation_distance_band'],as_index=False).agg(pixels=('pixels','sum'),changed=('changed','sum'),whitened=('whitened','sum'),print_like_pixels=('print_like_pixels','sum'),changed_print_like=('changed_print_like','sum'),whitened_print_like=('whitened_print_like','sum'))
for c,n in [('changed_rate','changed'),('whitened_rate','whitened')]:pooled[c]=pooled[n]/np.maximum(pooled.pixels,1)
pooled['whitened_print_like_rate']=pooled.whitened_print_like/np.maximum(pooled.print_like_pixels,1)
pooled.to_csv(TABLES/'authority_by_annotation_distance_pooled.csv',index=False)
print(page_df.sort_values('whitened_print_like',ascending=False).head(20).to_string(index=False))

## 5. Aggregate charts and multidimensional heatmaps

In [ ]:
sns.set_theme(style='whitegrid')
# 1. Authority-band burden.
auth=(pd.concat([pd.read_parquet(p,columns=['authority_band','changed_8plus','whitened_8plus','print_like_observable']) for p in PARTS.glob('*.parquet')],ignore_index=True).groupby('authority_band',as_index=False).agg(pixels=('changed_8plus','size'),changed=('changed_8plus','sum'),whitened=('whitened_8plus','sum'),print_like=('print_like_observable','sum')))
auth['changed_rate']=auth.changed/auth.pixels;auth['whitened_rate']=auth.whitened/auth.pixels
auth.to_csv(TABLES/'authority_band_summary.csv',index=False)
plt.figure(figsize=(9,5));q=auth.set_index('authority_band').reindex(['SEED','RING1','RING2','OTHER_PREPARED']).dropna().reset_index();x=np.arange(len(q));plt.bar(x-.18,q.changed_rate,width=.36,label='changed >=8');plt.bar(x+.18,q.whitened_rate,width=.36,label='whitened >=8');plt.xticks(x,q.authority_band);plt.ylabel('Fraction of band pixels');plt.title('Correction activity by mask-authority band');plt.legend();plt.tight_layout();plt.savefig(VISUAL/'authority_band_change_rates.png',dpi=180);plt.close()
# 2. 2D heatmap: authority x annotation distance.
def heat(value,title,path,fmt='.3f'):
    pivot=pooled.pivot(index='authority_band',columns='annotation_distance_band',values=value).reindex(['SEED','RING1','RING2','OTHER_PREPARED'])
    plt.figure(figsize=(12,5));sns.heatmap(pivot,annot=True,fmt=fmt,cmap='magma',linewidths=.5);plt.title(title);plt.xlabel('Distance to dark annotation proxy inside seed');plt.ylabel('Authority band');plt.tight_layout();plt.savefig(path,dpi=190);plt.close()
heat('whitened_print_like_rate','Printed-looking pixels whitened: authority band x annotation distance',VISUAL/'heatmap_whitened_print_like_rate.png')
heat('whitened_print_like','Count of whitened printed-looking pixels',VISUAL/'heatmap_whitened_print_like_count.png',fmt='.0f')
# 3. Page scatter: explicit dilation versus seed width.
page_df['collar_whitened_print_like']=page_df.ring1_whitened_print_like+page_df.ring2_whitened_print_like
plt.figure(figsize=(8,7));sns.scatterplot(data=page_df,x='seed_whitened_print_like',y='collar_whitened_print_like',hue='population',size='whitened_print_like',sizes=(20,260),alpha=.75);mx=max(page_df.seed_whitened_print_like.max(),page_df.collar_whitened_print_like.max(),1);plt.plot([0,mx],[0,mx],'k--',linewidth=1);plt.xlabel('Whitened printed-looking pixels inside seed');plt.ylabel('Whitened printed-looking pixels in rings 1+2');plt.title('Seed-width versus explicit-dilation damage signal by page');plt.tight_layout();plt.savefig(VISUAL/'page_seed_vs_collar_damage.png',dpi=180);plt.close()
# 4. Distribution versus annotation distance.
parts=[]
for p in PARTS.glob('*.parquet'):
    q=pd.read_parquet(p,columns=['distance_to_dark_seed_190','changed_8plus','whitened_8plus','print_like_observable','authority_band']);q=q[q.print_like_observable];q['distance_band']=band_label(q.distance_to_dark_seed_190,ANNOTATION_DISTANCE_BINS);parts.append(q.groupby(['authority_band','distance_band'],observed=True).agg(pixels=('changed_8plus','size'),whitened=('whitened_8plus','sum')).reset_index())
dist=pd.concat(parts).groupby(['authority_band','distance_band'],as_index=False).sum();dist['rate']=dist.whitened/dist.pixels;dist.to_csv(TABLES/'print_like_whitening_by_distance.csv',index=False)
plt.figure(figsize=(11,6));sns.lineplot(data=dist,x='distance_band',y='rate',hue='authority_band',marker='o');plt.ylabel('Whitened fraction of printed-looking pixels');plt.xlabel('Distance to dark seed annotation proxy');plt.title('Print-like whitening by annotation distance and authority band');plt.xticks(rotation=30);plt.tight_layout();plt.savefig(VISUAL/'print_like_whitening_by_annotation_distance.png',dpi=180);plt.close()
print('Visual summaries:',VISUAL)

## 6. Whole-page diagnostic overlays

In [ ]:
COL={'SEED':(0,200,0),'RING1':(0,215,255),'RING2':(0,0,255),'CHANGED':(255,0,255),'DARKCORE':(255,255,0),'CONF':(255,0,0),'REMOVE':(0,170,0),'AMBIG':(0,130,255),'PRESERVE':(255,100,0),'SAFE':(180,0,180),'M5':(0,0,220)}
def blend(gray,layers,alpha=.55):
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR);out=base.copy()
    for mask,bgr in layers:
        colour=np.zeros_like(base);colour[:]=bgr;out[mask]=cv2.addWeighted(base,1-alpha,colour,alpha,0)[mask]
    return out
def legend(image,title,items):
    h,w=image.shape[:2];banner=max(64,int(h*.04));can=np.full((h+banner,w,3),255,np.uint8);can[banner:]=image;scale=max(.42,min(.8,w/1800));cv2.putText(can,title,(12,int(banner*.38)),cv2.FONT_HERSHEY_SIMPLEX,scale,(25,25,25),1,cv2.LINE_AA);x=12;y=int(banner*.80)
    for lab,bgr in items:
        cv2.rectangle(can,(x,y-14),(x+18,y+4),bgr,-1);cv2.putText(can,lab,(x+24,y+3),cv2.FONT_HERSHEY_SIMPLEX,scale*.72,(25,25,25),1,cv2.LINE_AA);x+=max(145,int(len(lab)*8+55))
    return can
def colour_distance(gray,dist,valid,maxd=16):
    n=np.clip(dist/maxd,0,1);heat=cv2.applyColorMap((n*255).astype(np.uint8),cv2.COLORMAP_TURBO);base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR);out=base.copy();out[valid]=cv2.addWeighted(base,.42,heat,.58,0)[valid];return out

for _,r in roster.iterrows():
    key=f'{r.population}__{r.page}';page=str(r.page);z=page_inputs(key,page);observed=z['observed'];corrected=z['corrected'];prepared=z['prepared'];seed=z['seed'];ring1=z['ring1'];ring2=z['ring2'];confidence=z['confidence'];clean=z['clean'];reason=z['reason']
    changed=prepared&(np.abs(corrected.astype(np.int16)-observed.astype(np.int16))>=CHANGE_THRESHOLD);dark_core=seed&(observed<PRIMARY_DARK_THRESHOLD);dist=distance_to_core(dark_core)
    conf_core=(prepared&(confidence>=.80)) if confidence is not None else np.zeros(prepared.shape,bool)
    geometry=legend(blend(observed,[(seed,COL['SEED']),(ring1,COL['RING1']),(ring2,COL['RING2']),(changed,COL['CHANGED'])]),f'{key} — authority geometry and changed pixels',[('seed',COL['SEED']),('ring 1',COL['RING1']),('ring 2',COL['RING2']),('changed',COL['CHANGED'])])
    evidence=legend(blend(observed,[(dark_core,COL['DARKCORE']),(conf_core,COL['CONF']),(changed,COL['CHANGED'])]),f'{key} — annotation evidence and changed pixels',[('dark seed core',COL['DARKCORE']),('confidence >=0.80',COL['CONF']),('changed',COL['CHANGED'])])
    routes=legend(blend(observed,[(reason==1,COL['PRESERVE']),(reason==2,COL['CONF']),(reason==3,COL['M5']),(reason==4,COL['SAFE'])]),f'{key} — final runtime disposition',[('V7.8 preserve',COL['PRESERVE']),('strict PRINT',COL['CONF']),('M5',COL['M5']),('SAFE_REMOVE',COL['SAFE'])])
    distance=legend(colour_distance(observed,dist,prepared),f'{key} — distance to dark annotation proxy inside seed',[('near = blue', (255,0,0)),('far = red',(0,0,255))])
    d=PAGES/key;d.mkdir(parents=True,exist_ok=False);cv2.imwrite(str(d/'authority_overlay.png'),geometry);cv2.imwrite(str(d/'annotation_evidence_overlay.png'),evidence);cv2.imwrite(str(d/'route_overlay.png'),routes);cv2.imwrite(str(d/'annotation_distance_heatmap.png'),distance);cv2.imwrite(str(d/'observed.png'),observed);cv2.imwrite(str(d/'corrected.png'),corrected)
    if clean is not None:
        truth_print=cv2.dilate(printed_ink(clean).astype(np.uint8),np.ones((3,3),np.uint8))>0;truth_ann=seed&(observed<PRIMARY_DARK_THRESHOLD)&~truth_print;truth=legend(blend(observed,[(truth_print&(prepared),COL['PRESERVE']),(truth_ann,COL['SAFE']),(changed,COL['CHANGED'])]),f'{key} — POST-HOC clean-supported ownership',[('clean print vicinity',COL['PRESERVE']),('truth annotation proxy',COL['SAFE']),('changed',COL['CHANGED'])]);cv2.imwrite(str(d/'truth_annotation_overlay_posthoc.png'),truth)
    del z,observed,corrected,prepared,seed,ring1,ring2,confidence,clean,reason,changed,dark_core,dist,conf_core,geometry,evidence,routes,distance;gc.collect()
print('Per-page overlays complete.')

## 7. Ranked contact sheets and focused crops

In [ ]:
ranked=page_df.sort_values(['whitened_print_like','changed_print_like'],ascending=False).head(MAX_CONTACT_PAGES)
def contact(rows,name,files):
    if rows.empty:return
    fig,axes=plt.subplots(len(rows),len(files),figsize=(5*len(files),5.5*len(rows)),squeeze=False)
    for i,(_,r) in enumerate(rows.iterrows()):
        for j,(fn,title) in enumerate(files):
            p=PAGES/r.page_key/fn;im=cv2.imread(str(p),cv2.IMREAD_UNCHANGED)
            if im is not None and im.ndim==3:im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
            axes[i,j].imshow(im,cmap='gray' if im is not None and im.ndim==2 else None);axes[i,j].set_title(f'{r.page_key}\n{title}');axes[i,j].axis('off')
    plt.tight_layout();plt.savefig(VISUAL/f'{name}.png',dpi=125,bbox_inches='tight');plt.close(fig)
contact(ranked,'worst_page_geometry_review',[('observed.png','observed'),('corrected.png','corrected'),('authority_overlay.png','seed/rings/changes'),('annotation_distance_heatmap.png','annotation distance')])
contact(ranked,'worst_page_route_review',[('corrected.png','corrected'),('route_overlay.png','runtime routes'),('annotation_evidence_overlay.png','annotation evidence')])
# Focused crops: changed printed-looking connected regions, ranked by whitening mass.
candidates=[]
for _,r in page_df.sort_values('whitened_print_like',ascending=False).iterrows():
    part=pd.read_parquet(PARTS/f'{r.page_key}.parquet');hot=np.zeros((int(part.y.max())+1,int(part.x.max())+1),np.uint8);q=part[part.whitened_8plus&part.print_like_observable]
    if q.empty:continue
    hot[q.y.to_numpy(int),q.x.to_numpy(int)]=1;n,lab,stats,_=cv2.connectedComponentsWithStats(hot,8)
    for cid in range(1,n):
        x,y,w,h,a=stats[cid];mass=float(q.assign(cid=lab[q.y.to_numpy(int),q.x.to_numpy(int)]).loc[lambda x:x.cid==cid,'signed_change'].clip(lower=0).sum());candidates.append({'page_key':r.page_key,'component':cid,'x':x,'y':y,'w':w,'h':h,'area':a,'whitening_mass':mass})
crops=pd.DataFrame(candidates).sort_values(['whitening_mass','area'],ascending=False).head(MAX_FOCUS_CROPS);crops.to_csv(TABLES/'focus_crop_queue.csv',index=False)
if len(crops):
    fig,axes=plt.subplots(len(crops),4,figsize=(16,4.5*len(crops)),squeeze=False)
    for i,(_,r) in enumerate(crops.iterrows()):
        obs=read_gray(PAGES/r.page_key/'observed.png');cor=read_gray(PAGES/r.page_key/'corrected.png');H,W=obs.shape;x0=max(0,int(r.x)-CROP_MARGIN);y0=max(0,int(r.y)-CROP_MARGIN);x1=min(W,int(r.x+r.w)+CROP_MARGIN);y1=min(H,int(r.y+r.h)+CROP_MARGIN)
        ims=[obs[y0:y1,x0:x1],cor[y0:y1,x0:x1],cv2.cvtColor(cv2.imread(str(PAGES/r.page_key/'authority_overlay.png'))[y0:y1,x0:x1],cv2.COLOR_BGR2RGB),cv2.cvtColor(cv2.imread(str(PAGES/r.page_key/'annotation_distance_heatmap.png'))[y0:y1,x0:x1],cv2.COLOR_BGR2RGB)]
        titles=['observed','corrected','authority geometry','annotation distance']
        for j,(im,title) in enumerate(zip(ims,titles)):axes[i,j].imshow(im,cmap='gray' if im.ndim==2 else None);axes[i,j].set_title(f'{r.page_key} | {title}');axes[i,j].axis('off')
    plt.tight_layout();plt.savefig(VISUAL/'focused_print_loss_crops.png',dpi=150,bbox_inches='tight');plt.close(fig)
print('Contact sheets and focused crops:',VISUAL)

## 8. Diagnostic interpretation summary

In [ ]:
seed_damage=int(page_df.seed_whitened_print_like.sum());collar_damage=int((page_df.ring1_whitened_print_like+page_df.ring2_whitened_print_like).sum());total=seed_damage+collar_damage
# Split seed damage by distance from the dark annotation proxy.
seed_detail=[]
for p in PARTS.glob('*.parquet'):
    q=pd.read_parquet(p,columns=['authority_band','distance_to_dark_seed_190','whitened_8plus','print_like_observable']);q=q[(q.authority_band=='SEED')&q.whitened_8plus&q.print_like_observable];seed_detail.append(q)
seed_detail=pd.concat(seed_detail,ignore_index=True) if seed_detail else pd.DataFrame()
near=int((seed_detail.distance_to_dark_seed_190<2).sum()) if len(seed_detail) else 0
far=int((seed_detail.distance_to_dark_seed_190>=4).sum()) if len(seed_detail) else 0
interpretation=[]
if collar_damage>seed_damage:interpretation.append('Explicit dilation rings contain more whitened printed-looking pixels than the undilated seed.')
else:interpretation.append('The undilated seed contains at least as much whitened printed-looking material as the explicit dilation rings; reducing only radius-2 dilation may not solve the problem.')
if far>near:interpretation.append('Within the seed, more damaged printed-looking pixels lie at least 4 px from the dark annotation proxy than within 2 px, supporting excess seed width / fat-mask overreach.')
else:interpretation.append('Within-seed damage is concentrated near the dark annotation proxy; annotation/print ownership may matter more than excess mask width.')
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':'DIAGNOSTIC_AUDIT_COMPLETE','promotion_authority':False,'pages':len(page_df),'seed_whitened_print_like':seed_damage,'collar_whitened_print_like':collar_damage,'seed_share':seed_damage/max(total,1),'collar_share':collar_damage/max(total,1),'seed_damage_near_annotation_lt2px':near,'seed_damage_far_from_annotation_ge4px':far,'interpretation':interpretation,'confidence_maps_available_pages':int(page_df.confidence_available.sum()),'clean_truth_available_pages':int(page_df.clean_available.sum()),'notes':['Dark-seed distance is an observable annotation proxy, not truth.','Optional clean-derived maps are post-hoc diagnostics only.','No correction or mask was changed.']}
atomic_json(OUT/'notebook74_verdict.json',record)
lines=['NOTEBOOK 74 — MASK AUTHORITY, ANNOTATION DISTANCE AND PRINT-LOSS AUDIT','',f'Pages: {len(page_df)}',f'Seed whitened printed-looking pixels: {seed_damage}',f'Dilation-ring whitened printed-looking pixels: {collar_damage}',f'Within-seed damaged pixels <2 px from annotation proxy: {near}',f'Within-seed damaged pixels >=4 px from annotation proxy: {far}',f'Confidence maps available: {record["confidence_maps_available_pages"]}',f'Clean truth available for post-hoc diagnostics: {record["clean_truth_available_pages"]}','','INTERPRETATION']+[f'- {x}' for x in interpretation]+['','No mask, route, probability or corrected page was changed.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines));print('\n'.join(lines))

## 9. Curated final upload package

In [ ]:
required=[OUT/'executive_summary.txt',OUT/'notebook74_verdict.json',INPUTS/'source_identity.json',AUDIT/'input_resolution.csv',TABLES/'page_summary.csv',TABLES/'authority_band_summary.csv',TABLES/'authority_by_annotation_distance_pooled.csv',TABLES/'print_like_whitening_by_distance.csv',TABLES/'focus_crop_queue.csv',VISUAL/'authority_band_change_rates.png',VISUAL/'heatmap_whitened_print_like_rate.png',VISUAL/'page_seed_vs_collar_damage.png',VISUAL/'print_like_whitening_by_annotation_distance.png',VISUAL/'worst_page_geometry_review.png',VISUAL/'worst_page_route_review.png']
if (VISUAL/'focused_print_loss_crops.png').exists():required.append(VISUAL/'focused_print_loss_crops.png')
for p in required:require(p.exists(),f'Missing package item: {p}')
manifest=[]
for src in required:
    dst=UPLOAD/src.name;shutil.copy2(src,dst);manifest.append({'file':dst.name,'source':str(src),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 74 read-only mask authority, annotation-distance and print-loss audit. Start with executive_summary.txt, the heatmaps, and worst-page contact sheets. Pixel-level parquet parts remain in the run directory.\n')
with zipfile.ZipFile(UPLOAD/'notebook74_mask_annotation_distance_audit_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p.suffix!='.zip':z.write(p,p.name)
print('Bundle:',UPLOAD/'notebook74_mask_annotation_distance_audit_bundle.zip')

## Reading the result

Use these visuals first:

1. `heatmap_whitened_print_like_rate.png` — separates seed-width effects from explicit dilation effects across annotation distance.
2. `page_seed_vs_collar_damage.png` — shows whether page-level damage is dominated by the reviewed seed or dilation rings.
3. `print_like_whitening_by_annotation_distance.png` — tests whether whitening rises with clearance from annotation evidence.
4. `worst_page_geometry_review.png` — observed, corrected, authority geometry and annotation-distance maps side by side.
5. `focused_print_loss_crops.png` — close visual inspection of the strongest candidate defects.

A reduction in dilation is supported only if ring 1/2 damage dominates after controlling for annotation distance. If damage is already substantial inside the seed and far from annotation evidence, the seed-mask construction or reviewer brush width is the more direct problem.